# Common · 03 — Deep models, training and evaluation library

Requires `common/00_config.ipynb` and `common/01_signal_lib.ipynb`. MST-ResFLNet-style branches, attention fusion, focal loss, training loop, metrics, JPEG-like compression of time-frequency images.

## `nets`

Phase 8/11 â€” tri-branch temporal-spectral-wavelet network with adaptive attention fusion.

The 2-D branches follow MST-ResFLNet (IEEE, 2025): a multi-branch *pyramid spectrum stem*
(parallel 3x3 / 5x5 / 7x7 kernels) replacing the single conv1, squeeze-and-excitation
attention, residual stages, and *temporal attention pooling* (average over frequency, then
attention weights over time frames) instead of global average pooling.

Fusion (Phase 11):  F = alpha_T F_T + alpha_S F_S + alpha_W F_W,  alpha = softmax over the
branches enabled by the per-recording fusion action (disabled branches get -inf logits).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

BRANCHES = ["T", "S", "W"]


class SE(nn.Module):
    def __init__(self, c, r=8, dim=2):
        super().__init__()
        self.dim = dim
        self.fc = nn.Sequential(nn.Linear(c, max(4, c // r)), nn.ReLU(), nn.Linear(max(4, c // r), c), nn.Sigmoid())

    def forward(self, x):
        s = x.mean(dim=tuple(range(2, x.dim())))
        w = self.fc(s)
        return x * w.view(*w.shape, *([1] * (x.dim() - 2)))


class TemporalAttentionPool(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.att = nn.Sequential(nn.Conv1d(c, c // 2, 1), nn.Tanh(), nn.Conv1d(c // 2, 1, 1))

    def forward(self, x):
        w = torch.softmax(self.att(x), dim=-1)
        return (x * w).sum(-1), w.squeeze(1)


class Res2d(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)
        self.se = SE(cout)
        self.sc = nn.Sequential() if stride == 1 and cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        y = self.se(self.b2(self.c2(y)))
        return F.relu(y + self.sc(x))


class Res1d(nn.Module):
    def __init__(self, cin, cout, stride, k=7):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, k, stride, k // 2, bias=False)
        self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, k, 1, k // 2, bias=False)
        self.b2 = nn.BatchNorm1d(cout)
        self.se = SE(cout)
        self.sc = nn.Sequential() if stride == 1 and cin == cout else nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))

    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        y = self.se(self.b2(self.c2(y)))
        return F.relu(y + self.sc(x))


class PyramidStem2d(nn.Module):
    def __init__(self, cout=48):
        super().__init__()
        c = cout // 3
        self.paths = nn.ModuleList([nn.Conv2d(1, c, k, 1, k // 2, bias=False) for k in (3, 5, 7)])
        self.bn = nn.BatchNorm2d(3 * c)
        self.se = SE(3 * c)

    def forward(self, x):
        return self.se(F.relu(self.bn(torch.cat([p(x) for p in self.paths], 1))))


class TFBranch(nn.Module):
    def __init__(self, d=128, pyramid=True):
        super().__init__()
        self.inorm = nn.InstanceNorm2d(1, affine=True)
        self.stem = PyramidStem2d(48) if pyramid else nn.Sequential(nn.Conv2d(1, 48, 3, 1, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU())
        self.stages = nn.Sequential(Res2d(48, 64, 2), Res2d(64, 96, 2), Res2d(96, d, 2))
        self.pool = TemporalAttentionPool(d)

    def forward(self, x):
        h = self.stages(self.stem(self.inorm(x)))
        f, w = self.pool(h.mean(2))
        return f, w


class TemporalBranch(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.paths = nn.ModuleList([nn.Conv1d(1, 16, k, 4, k // 2, bias=False) for k in (9, 31, 63)])
        self.bn = nn.BatchNorm1d(48)
        self.se = SE(48)
        self.stages = nn.Sequential(Res1d(48, 64, 2), Res1d(64, 96, 2), Res1d(96, d, 2), Res1d(d, d, 2))
        self.pool = TemporalAttentionPool(d)

    def forward(self, x):
        h = self.se(F.relu(self.bn(torch.cat([p(x) for p in self.paths], 1))))
        return self.pool(self.stages(h))


class FusionNet(nn.Module):
    def __init__(self, d=128, fusion="attention", branches=("T", "S", "W"), dropout=0.3):
        super().__init__()
        self.fusion, self.branches = fusion, list(branches)
        self.enc = nn.ModuleDict()
        if "T" in branches:
            self.enc["T"] = TemporalBranch(d)
        if "S" in branches:
            self.enc["S"] = TFBranch(d)
        if "W" in branches:
            self.enc["W"] = TFBranch(d)
        self.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
        self.concat_proj = nn.Linear(d * len(self.branches), d)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, batch, return_details=False):
        mask = batch["mask"][:, [BRANCHES.index(b) for b in self.branches]]
        feats, tatt = [], {}
        for b in self.branches:
            f, w = self.enc[b](batch[b])
            feats.append(f)
            tatt[b] = w
        out, alpha = self.fuse(torch.stack(feats, 1), mask)
        if return_details:
            return out, {"alpha": alpha, "time_attention": tatt}
        return out

    def fuse(self, Fm, mask):
        if self.fusion == "attention":
            logits = self.score(Fm).squeeze(-1).masked_fill(~mask, float("-inf"))
            alpha = torch.softmax(logits, 1)
            fused = (alpha.unsqueeze(-1) * Fm).sum(1)
        else:
            alpha = mask.float() / mask.float().sum(1, keepdim=True)
            fused = self.concat_proj((Fm * mask.unsqueeze(-1)).flatten(1))
        return self.head(fused).squeeze(-1), alpha


class FocalBCE(nn.Module):
    def __init__(self, pos_weight: float, gamma=2.0, focal_weight=1.0):
        super().__init__()
        self.pw, self.g, self.fw = pos_weight, gamma, focal_weight

    def forward(self, logit, y):
        w = torch.where(y > 0.5, torch.full_like(y, self.pw), torch.ones_like(y))
        bce = F.binary_cross_entropy_with_logits(logit, y, reduction="none")
        pt = torch.exp(-bce)
        return (w * bce).mean() + self.fw * (w * (1 - pt) ** self.g * bce).mean()


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## `deep`

Deep training / inference where every recording (or every crop) has its own configuration.

A configuration c = (wavelet, level, band, window, fusion) decides the inputs of each branch:
T = the raw window, S = STFT rows inside the band, W = wavelet-packet (wavelet, level) rows
inside the band, computed per crop. Batches share one window length; everything else may
differ per item, which is what lets one network serve every configuration (supernet).

In [ ]:
import math
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pywt
import torch
import torch.nn.functional as F

BR = ["T", "S", "W"]


class RecordStore:
    def __init__(self, signals, configs, fs: int, deep_cfg: dict):
        self.x = [s.astype(np.float32) for s in signals]
        self.cfg = configs
        self.fs, self.d = fs, deep_cfg

    def win(self, i, c=None):
        return int((c or self.cfg[i])["window_s"] * self.fs)


def _crop(x, start, n):
    seg = x[start: start + n]
    if len(seg) < n:
        seg = np.pad(seg, (0, n - len(seg)))
    return seg


def spectral_images(T: torch.Tensor, bands, d: dict, fs: int) -> torch.Tensor:
    spec = torch.stft(T, d["n_fft"], d["hop"], window=torch.hann_window(d["n_fft"], device=T.device),
                      return_complex=True, center=False).abs() ** 2
    spec = torch.log1p(spec * 10)
    freqs = torch.linspace(0, fs / 2, spec.shape[1], device=T.device)
    out = torch.empty(len(T), 1, d["spec_rows"], d["tf_frames"], device=T.device)
    groups = {}
    for j, b in enumerate(bands):
        groups.setdefault(tuple(b), []).append(j)
    for (lo, hi), js in groups.items():
        m = (freqs >= lo) & (freqs <= hi)
        if m.sum() < 2:
            k = int(torch.argmin((freqs - (lo + hi) / 2).abs()))
            m = torch.zeros_like(m)
            m[max(0, k - 1): k + 1] = True
        out[js] = F.interpolate(spec[js][:, m][:, None], (d["spec_rows"], d["tf_frames"]), mode="bilinear", align_corners=False)
    return out


def wavelet_images(crops: np.ndarray, cfgs, d: dict, fs: int, device) -> torch.Tensor:
    out = torch.empty(len(crops), 1, d["wav_rows"], d["tf_frames"], device=device)
    by_wav = {}
    for j, c in enumerate(cfgs):
        by_wav.setdefault(c["wavelet"], []).append(j)
    for wv, js in by_wav.items():
        maxL = max(cfgs[j]["level"] for j in js)
        wp = pywt.WaveletPacket(crops[js], wv, mode="periodization", maxlevel=maxL, axis=-1)
        levels = {}
        for pos, j in enumerate(js):
            levels.setdefault((cfgs[j]["level"], tuple(cfgs[j]["band"])), []).append((pos, j))
        cache = {}
        for (L, band), members in levels.items():
            if L not in cache:
                cache[L] = np.stack([nd.data for nd in wp.get_level(L, order="freq")], axis=1)
            M = cache[L][[p for p, _ in members]][:, band_rows(cache[L].shape[1], band, fs)]
            t = torch.from_numpy(log_compress(M)).to(device)[:, None]
            out[[j for _, j in members]] = F.interpolate(t, (d["wav_rows"], d["tf_frames"]), mode="bilinear", align_corners=False)
    return out


def make_batch(store: RecordStore, items, device, branches=("T", "S", "W"), img_transform=None, configs=None):
    cfgs = configs or [store.cfg[i] for i, _ in items]
    n = store.win(items[0][0], cfgs[0])
    crops = np.stack([_crop(store.x[i], s, n) for i, s in items])
    T = torch.from_numpy(crops).to(device)
    out = {"T": T[:, None], "mask": torch.tensor([[b in c["fusion"] for b in BR] for c in cfgs], device=device)}
    if "S" in branches:
        out["S"] = spectral_images(T, [c["band"] for c in cfgs], store.d, store.fs)
    if "W" in branches:
        out["W"] = wavelet_images(crops, cfgs, store.d, store.fs, device)
    if img_transform is not None:
        for b in ("S", "W"):
            if b in out:
                out[b] = img_transform(out[b])
    return out


def random_config(space, rng, window_idx=None) -> dict:
    code = [int(rng.integers(d)) for d in space.dims]
    if window_idx is not None:
        code[3] = window_idx
    return config_dict(space, space.encode(code))


def train_batches(store, idx, crops, batch_size, rng, space=None):
    if space is not None:
        pool = np.repeat(np.asarray(idx), crops)
        rng.shuffle(pool)
        out = []
        for j in range(0, len(pool), batch_size):
            w = int(rng.integers(space.dims[3]))
            cfgs = [random_config(space, rng, w) for _ in pool[j: j + batch_size]]
            n = store.win(0, cfgs[0])
            items = [(int(i), int(rng.integers(0, max(1, len(store.x[i]) - n + 1)))) for i in pool[j: j + batch_size]]
            out.append((items, cfgs))
        return out
    buckets = {}
    for i in idx:
        n = store.win(i)
        L = len(store.x[i])
        for _ in range(crops):
            buckets.setdefault(n, []).append((i, int(rng.integers(0, max(1, L - n + 1)))))
    batches = []
    for items in buckets.values():
        rng.shuffle(items)
        batches += [(items[j: j + batch_size], None) for j in range(0, len(items), batch_size)]
    rng.shuffle(batches)
    return batches


def prefetched(jobs, depth=4):
    with ThreadPoolExecutor(depth) as ex:
        futs = [ex.submit(f, *a) for f, a in jobs[:depth]]
        for k in range(len(jobs)):
            res = futs[k].result()
            if k + depth < len(jobs):
                f, a = jobs[k + depth]
                futs.append(ex.submit(f, *a))
            yield res


def eval_items(store, idx, max_windows=24):
    buckets = {}
    for i in idx:
        n = store.win(i)
        L = len(store.x[i])
        starts = np.arange(0, max(1, L - n + 1), max(1, n // 2))
        if len(starts) > max_windows:
            starts = starts[np.linspace(0, len(starts) - 1, max_windows).astype(int)]
        for s in starts:
            buckets.setdefault(n, []).append((i, int(s)))
    return buckets


@torch.no_grad()
def predict(model, store, idx, device, batch_size=128, img_transform=None, details=False):
    model.eval()
    sums = np.zeros(len(store.x))
    cnt = np.zeros(len(store.x))
    alphas = {i: [] for i in idx}
    chunks = [items[j: j + batch_size] for items in eval_items(store, idx).values() for j in range(0, len(items), batch_size)]
    jobs = [(make_batch, (store, chunk, device, model.branches, img_transform)) for chunk in chunks]
    for chunk, batch in zip(chunks, prefetched(jobs)):
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.type == "cuda"):
            if details:
                logit, det = model(batch, return_details=True)
            else:
                logit = model(batch)
        p = torch.sigmoid(logit.float()).cpu().numpy()
        np.add.at(sums, [i for i, _ in chunk], p)
        np.add.at(cnt, [i for i, _ in chunk], 1)
        if details:
            a = det["alpha"].float().cpu().numpy()
            for (i, _), ai in zip(chunk, a):
                alphas[i].append(ai)
    probs = sums[idx] / np.maximum(cnt[idx], 1)
    if details:
        return probs, {i: np.mean(v, 0) for i, v in alphas.items() if v}
    return probs


def fit(model, store, y, tr_idx, va_idx, deep_cfg, device, seed=0, log=print, init_state=None, lr=None, epochs=None,
        space=None, patience=None):
    from sklearn.metrics import roc_auc_score

    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    model.to(device)
    if init_state is not None:
        model.load_state_dict(init_state)
    lr = lr or deep_cfg["lr"]
    epochs = epochs or deep_cfg["epochs"]
    patience = patience or deep_cfg["patience"]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=deep_cfg["weight_decay"])
    pos = y[tr_idx].mean()
    crit = FocalBCE(pos_weight=(1 - pos) / pos, gamma=deep_cfg["focal_gamma"])
    steps_per_epoch = math.ceil(len(tr_idx) * deep_cfg["crops_per_record"] / deep_cfg["batch_size"]) + 4
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * steps_per_epoch, pct_start=0.15)
    best, best_state, bad, hist = -1, None, 0, []
    yt = torch.from_numpy(y.astype(np.float32))
    for ep in range(epochs):
        model.train()
        t0, tot, nb = time.time(), 0.0, 0
        plan = train_batches(store, tr_idx, deep_cfg["crops_per_record"], deep_cfg["batch_size"], rng, space)[:steps_per_epoch]
        jobs = [(make_batch, (store, items, device, model.branches, None, cfgs)) for items, cfgs in plan]
        for (items, cfgs), batch in zip(plan, prefetched(jobs)):
            target = yt[[i for i, _ in items]].to(device)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.type == "cuda"):
                logit = model(batch)
            loss = crit(logit.float(), target)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            sched.step()
            tot += loss.item()
            nb += 1
        pv = predict(model, store, va_idx, device)
        auc = roc_auc_score(y[va_idx], pv)
        hist.append({"epoch": ep + 1, "loss": tot / max(nb, 1), "val_auroc": auc, "sec": time.time() - t0})
        log(f"  ep {ep + 1:02d} loss {tot / max(nb, 1):.4f} val AUROC {auc:.4f} ({time.time() - t0:.0f}s)")
        if auc > best:
            best, bad = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, hist


@torch.no_grad()
def supernet_config_probs(model, signals, space, cfg, device, seg_s=4.0, chunk=768):
    model.eval()
    d, fs, A = cfg["deep"], cfg["preprocessing"]["fs"], cfg["actions"]
    codes = np.array([space.decode(c) for c in range(space.n)])
    fmask = torch.tensor([[b in f for b in BR] for f in A["fusion"]], device=device)
    out = np.zeros((space.n, len(signals)), np.float32)
    nW, nL, nB = len(A["wavelet"]), len(A["level"]), len(A["band"])
    ac = torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.type == "cuda")
    for r, x in enumerate(signals):
        L = int(seg_s * fs)
        a = max(0, (len(x) - L) // 2)
        seg = x[a: a + L]
        for ti, w in enumerate(A["window_s"]):
            n = int(w * fs)
            crops = np.stack([_crop(seg, s, n) for s in range(0, max(1, len(seg) - n + 1), n)])
            nc = len(crops)
            T = torch.from_numpy(crops).to(device)
            with ac:
                FT = model.enc["T"](T[:, None])[0].float()
                FS = torch.stack([model.enc["S"](spectral_images(T, [b] * nc, d, fs))[0].float() for b in A["band"]])
                wps = [pywt.WaveletPacket(crops, wv, mode="periodization", maxlevel=max(A["level"]), axis=-1) for wv in A["wavelet"]]
                Wimg = torch.empty(nW, nL, nB, nc, 1, d["wav_rows"], d["tf_frames"], device=device)
                for li, Lv in enumerate(A["level"]):
                    M = torch.from_numpy(np.stack([np.stack([nd.data for nd in wp.get_level(Lv, order="freq")], axis=1) for wp in wps])).to(device)
                    M = torch.log1p(M.abs() * 10.0)
                    for bi, b in enumerate(A["band"]):
                        sub = M[:, :, band_rows(M.shape[2], b, fs)].reshape(nW * nc, 1, -1, M.shape[-1])
                        Wimg[:, li, bi] = F.interpolate(sub, (d["wav_rows"], d["tf_frames"]), mode="bilinear", align_corners=False).view(nW, nc, 1, d["wav_rows"], d["tf_frames"])
                Wimg = Wimg.view(-1, 1, d["wav_rows"], d["tf_frames"])
                FW = torch.cat([model.enc["W"](Wimg[j: j + chunk])[0].float() for j in range(0, len(Wimg), chunk)])
                FW = FW.view(nW, nL, nB, nc, -1)
                sel = np.where(codes[:, 3] == ti)[0]
                cw, cl, cb, cf = (torch.from_numpy(codes[sel, k]).to(device) for k in (0, 1, 2, 4))
                Fm = torch.stack([FT.expand(len(sel), nc, -1), FS[cb], FW[cw, cl, cb]], dim=2)
                m = fmask[cf][:, None, :].expand(-1, nc, -1)
                logit, _ = model.fuse(Fm.reshape(-1, 3, Fm.shape[-1]), m.reshape(-1, 3))
            out[sel, r] = torch.sigmoid(logit.float()).view(len(sel), nc).mean(1).cpu().numpy()
    return out


def best_threshold(y, p):
    from sklearn.metrics import f1_score

    grid = np.unique(np.quantile(p, np.linspace(0.02, 0.98, 97)))
    scores = [f1_score(y, (p >= t).astype(int), average="macro") for t in grid]
    return float(grid[int(np.argmax(scores))])


def _dct_mat(n=8, device="cpu"):
    k = torch.arange(n, device=device).float()
    D = torch.cos(math.pi * (2 * k[None] + 1) * k[:, None] / (2 * n)) * math.sqrt(2 / n)
    D[0] /= math.sqrt(2)
    return D


JPEG_Q = torch.tensor([
    [16, 11, 10, 16, 24, 40, 51, 61], [12, 12, 14, 19, 26, 58, 60, 55], [14, 13, 16, 24, 40, 57, 69, 56],
    [14, 17, 22, 29, 51, 87, 80, 62], [18, 22, 37, 56, 68, 109, 103, 77], [24, 35, 55, 64, 81, 104, 113, 92],
    [49, 64, 78, 87, 103, 121, 120, 101], [72, 92, 95, 98, 112, 100, 103, 99]], dtype=torch.float32)


def jpeg_like(img: torch.Tensor, quality: int, stats: dict | None = None) -> torch.Tensor:
    B, C, Hh, Ww = img.shape
    dev = img.device
    lo = img.amin(dim=(2, 3), keepdim=True)
    hi = img.amax(dim=(2, 3), keepdim=True)
    x = (img - lo) / (hi - lo + 1e-8) * 255 - 128
    x = F.pad(x, (0, (-Ww) % 8, 0, (-Hh) % 8), mode="replicate")
    D = _dct_mat(8, dev)
    coef = D @ x.unfold(2, 8, 8).unfold(3, 8, 8) @ D.T
    s = 5000 / quality if quality < 50 else 200 - 2 * quality
    Q = torch.clamp(torch.floor((JPEG_Q.to(dev) * s + 50) / 100), min=1)
    q = torch.round(coef / Q)
    rec = D.T @ (q * Q) @ D
    h, w = rec.shape[2], rec.shape[3]
    rec = rec.permute(0, 1, 2, 4, 3, 5).reshape(B, C, h * 8, w * 8)[:, :, :Hh, :Ww]
    if stats is not None:
        _, cnt = torch.unique(q, return_counts=True)
        pr = cnt.float() / cnt.sum()
        stats["bits_per_pixel"] = float(-(pr * torch.log2(pr)).sum())
        mse = F.mse_loss((rec + 128) / 255, (x[:, :, :Hh, :Ww] + 128) / 255).item()
        stats["psnr_db"] = 10 * math.log10(1 / max(mse, 1e-12))
    return (rec + 128) / 255 * (hi - lo) + lo


def config_dict(space, c: int) -> dict:
    v = space.values(int(c))
    return {"wavelet": v["wavelet"], "level": v["level"], "band": tuple(v["band"]), "window_s": v["window_s"], "fusion": v["fusion"]}


def load_deep_data(P):
    meta = pd.read_csv(P["processed"] / "proxy_records.csv")
    ctx = pd.read_csv(P["processed"] / "contexts.csv")
    meta = meta.merge(ctx, on="record_id", how="left")
    sig = np.load(P["segments"] / "signals.npz")
    return meta, [sig[r].astype(np.float32) for r in meta.record_id]


def policy_configs(meta, space, policy, K):
    return [config_dict(space, policy[k]) for k in meta[f"ctx_K{K}"].values]


def run_experiment(name, meta, signals, configs, cfg, P, device, seeds=(0,), branches=("T", "S", "W"),
                   fusion="attention", log=print, overwrite=False, init_from=None, lr=None, epochs=None):
    out_dir = P["models"] / name
    out_dir.mkdir(parents=True, exist_ok=True)
    fs = cfg["preprocessing"]["fs"]
    y = meta.y.values.astype(np.float32)
    idx = {s: np.where(meta.split == s)[0] for s in ("train", "val", "test")}
    store = RecordStore(signals, configs, fs, cfg["deep"])
    rows = []
    for seed in seeds:
        if (out_dir / f"summary_s{seed}.csv").exists() and not overwrite:
            log(f"[{name} s{seed}] cached")
            rows.append(load_summary(name, seed, P))
            continue
        seed_everything(seed)
        model = FusionNet(fusion=fusion, branches=branches)
        init = None
        if init_from:
            src = P["models"] / init_from / f"model_s{seed}.pt"
            init = torch.load(src if src.exists() else P["models"] / init_from / "model_s0.pt", map_location="cpu")
        t0 = time.time()
        model, hist = fit(model, store, y, idx["train"], idx["val"], cfg["deep"], device, seed=seed, log=log,
                          init_state=init, lr=lr, epochs=epochs)
        train_min = (time.time() - t0) / 60
        torch.save(model.state_dict(), out_dir / f"model_s{seed}.pt")
        pd.DataFrame(hist).to_csv(out_dir / f"history_s{seed}.csv", index=False)
        pv = predict(model, store, idx["val"], device)
        torch.cuda.synchronize()
        t1 = time.time()
        pt = predict(model, store, idx["test"], device)
        torch.cuda.synchronize()
        infer_ms = (time.time() - t1) / len(idx["test"]) * 1000
        thr = best_threshold(y[idx["val"]].astype(int), pv)
        pd.concat([pd.DataFrame({"record_id": meta.record_id.values[idx[s]], "split": s, "y": y[idx[s]].astype(int), "prob": p})
                   for s, p in (("val", pv), ("test", pt))]).to_csv(out_dir / f"preds_s{seed}.csv", index=False)
        mv = binary_metrics(y[idx["val"]], pv, thr)
        mt = binary_metrics(y[idx["test"]], pt, thr)
        row = {"model": name, "seed": seed, "threshold": thr, "params": n_params(model), "train_min": train_min,
               "infer_ms_per_recording": infer_ms, "epochs": len(hist),
               **{f"val_{k}": v for k, v in mv.items()}, **{f"test_{k}": v for k, v in mt.items()}}
        pd.DataFrame([row]).to_csv(out_dir / f"summary_s{seed}.csv", index=False)
        log(f"[{name} s{seed}] val AUROC {mv['auroc']:.3f} | test macro-F1 {mt['macro_f1']:.3f} AUROC {mt['auroc']:.3f} ({train_min:.1f} min)")
        rows.append(row)
        del model
        torch.cuda.empty_cache()
    return rows


def load_summary(name, seed, P) -> dict:
    return pd.read_csv(P["models"] / name / f"summary_s{seed}.csv").iloc[0].to_dict()


def load_model(name, seed, cfg, P, branches=("T", "S", "W"), fusion="attention"):
    m = FusionNet(fusion=fusion, branches=branches)
    m.load_state_dict(torch.load(P["models"] / name / f"model_s{seed}.pt", map_location="cpu"))
    return m

In [ ]:
print("loaded 03_deep_lib")